In [2]:
import os
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt 

# ignite
from ignite.engine import Engine, create_supervised_trainer, create_supervised_evaluator, Events
from ignite.handlers import ModelCheckpoint, global_step_from_engine
from ignite.handlers import EarlyStopping

In [3]:
data = "/home/jovyan/DADOS-DIVIDIDOS"
feature_extract= True

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
        transforms.RandomVerticalFlip(),
        transforms.RandomPerspective(distortion_scale=0.2, p=0.5, interpolation=3, fill=0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),  # Aumenta para evitar perda de informação
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

image_datasets = {x: datasets.ImageFolder(os.path.join(data, x), data_transforms[x]) for x in ['train', 'val','test']}


/opt/conda/lib/python3.10/site-packages/torchvision/transforms/transforms.py:768: UserWarning: Argument 'interpolation' of type int is deprecated since 0.13 and will be removed in 0.15. Please use InterpolationMode enum.
  warnings.warn(


In [4]:
# Extração de features + Congelamento dos parâmetros
def set_parameter_requires_grad(model, feature_extracting):
    if feature_extracting:
        for param in model.parameters():
            param.requires_grad = False

Best hyperparameters : {'dropout1': 0.332301792028777, 'dropout2': 0.2705522735661898, 'activation': 'ReLU', 'batch_size': 128, 'num_neurons_in': 512, 'num_neurons_out': 1024, 'optimizer': 'Adam', 'lr': 0.0013912900114223126}

In [5]:
dropout_rate1 = 0.332301792028777
dropout_rate2 = 0.2705522735661898
batch_size = 128
num_neurons_in = 512
num_neurons_out = 1024
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

In [6]:
import os
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models, transforms
from torch.utils.data import Subset
from sklearn.model_selection import StratifiedKFold
from ignite.metrics import Accuracy, Loss
from ignite.engine import Events, Engine, create_supervised_evaluator
from ignite.handlers import ModelCheckpoint, EarlyStopping
from ignite.contrib.handlers.param_scheduler import LRScheduler
from torch.optim.lr_scheduler import StepLR


def create_model():
    model = models.vgg19(pretrained=False)
    
    if feature_extract:
        for param in model.parameters():
            param.requires_grad = False
    
    set_parameter_requires_grad(model, feature_extract)

    model_vgg19 = "/home/jovyan/models/VggNet19-model-96.pth"
    state_dict = torch.load(model_vgg19)

    del state_dict['classifier.6.weight']
    del state_dict['classifier.6.bias']

    model.load_state_dict(state_dict, strict=False)

    model.classifier = nn.Sequential(
        nn.Linear(in_features=25088, out_features=num_neurons_out, bias=True),
        nn.ReLU(inplace=True),
        nn.Dropout(p=dropout_rate1, inplace=False),
        nn.Linear(in_features=num_neurons_out, out_features=num_neurons_in, bias=True),
        nn.ReLU(inplace=True),
        nn.Dropout(p=dropout_rate2, inplace=False),
        nn.Linear(in_features=num_neurons_in, out_features=2, bias=True),  # Saída final para 2 classes
        nn.Softmax(dim=1)
    )
    
    return model.to(device)

def train_step(engine, batch):
    model.train()
    inputs, labels = batch[0].to(device), batch[1].to(device)
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    return loss.item()

def validation_step(engine, batch):
    model.eval()
    with torch.no_grad():
        inputs, labels = batch[0].to(device), batch[1].to(device)
        outputs = model(inputs)
        return outputs, labels

n_splits = 10
train_labels = np.array([y for _, y in image_datasets['train']])
skf = StratifiedKFold(n_splits=n_splits, shuffle=True)

val_metrics = {
    "accuracy": Accuracy(),
    "loss": Loss(criterion)
}

def score_function(engine):
    return engine.state.metrics["accuracy"]

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(train_labels)), train_labels)):
    print(f'Fold {fold+1}/{n_splits}')
    
    # Cria um novo modelo para cada fold
    model = create_model()
    
    train_subset = Subset(image_datasets['train'], train_idx)
    val_subset = Subset(image_datasets['train'], val_idx)
    
    train_loader = torch.utils.data.DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_subset, batch_size=batch_size, shuffle=False)

    # Reinicializa o otimizador e scheduler para cada fold
    params_to_update = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(params_to_update, lr=0.001)
    torch_lr_scheduler = StepLR(optimizer, step_size=10, gamma=0.1)
    scheduler = LRScheduler(torch_lr_scheduler)
    

    trainer = Engine(train_step)
    evaluator = Engine(validation_step)
    train_evaluator = create_supervised_evaluator(model, metrics=val_metrics, device=device)

    Accuracy().attach(evaluator, 'accuracy')
    Loss(criterion).attach(evaluator, 'loss')
    Accuracy().attach(train_evaluator, 'accuracy')
    Loss(criterion).attach(train_evaluator, 'loss')

    train_accs = []
    val_accs = []
    train_losses = []
    val_losses = []
    
    @trainer.on(Events.STARTED)
    def start_message():
        print(f"Start training fold {fold+1}!")
        
        with open("result_vgg19_kfold.txt", 'a', encoding='utf-8') as file:
            file.write(f"Start training fold {fold+1}! \n\n")
            
        
    @trainer.on(Events.EPOCH_COMPLETED)
    def run_train_validation():
        train_evaluator.run(train_loader)

    @trainer.on(Events.EPOCH_COMPLETED)
    def run_validation():
        evaluator.run(val_loader)
        
    @trainer.on(Events.EPOCH_COMPLETED)
    def print_lr():
        print(f"Learning rate atual: {optimizer.param_groups[0]['lr']}")
        with open("result_vgg19_kfold.txt", 'a', encoding='utf-8') as file:
            file.write(f"Learning rate atual: {optimizer.param_groups[0]['lr']}\n\n")
    
    @train_evaluator.on(Events.COMPLETED)
    def log_train_results():
        metrics = train_evaluator.state.metrics
        train_acc = metrics['accuracy']
        train_loss = metrics['loss']
        train_accs.append(train_acc)
        train_losses.append(train_loss)
        print(f"Fold {fold+1} - Epoch {trainer.state.epoch} - Training Accuracy: {train_acc:.3f}, Loss: {train_loss:.3f}")
        
        with open("result_vgg19_kfold.txt", 'a', encoding='utf-8') as file:
            file.write(f"Fold {fold+1} - Epoch {trainer.state.epoch} - Training Accuracy: {train_acc:.3f}, Loss: {train_loss:.3f}\n\n")

    @evaluator.on(Events.COMPLETED)
    def log_validation_results():
        metrics = evaluator.state.metrics
        val_acc = metrics['accuracy']
        val_loss = metrics['loss']
        val_accs.append(val_acc)
        val_losses.append(val_loss)
        
        print(f"Fold {fold+1} - Epoch: {trainer.state.epoch} - Validation Accuracy: {val_acc:.3f}, Loss: {val_loss:.3f}")
        with open("result_vgg19_kfold.txt", 'a', encoding='utf-8') as file:
            file.write(f"Fold {fold+1} - Epoch: {trainer.state.epoch} - Validation Accuracy: {val_acc:.3f}, Loss: {val_loss:.3f}\n\n")

    handler = ModelCheckpoint(
        dirname=f'models_fold_{fold+1}',
        filename_prefix='best',
        n_saved=1,
        create_dir=True,
        score_function=score_function,
        score_name="val_acc",
        require_empty=False
    )
    evaluator.add_event_handler(Events.COMPLETED, handler, {'model': model})

    es_handler = EarlyStopping(patience=50, score_function=score_function, trainer=trainer)
    evaluator.add_event_handler(Events.COMPLETED, es_handler)

    trainer.run(train_loader, max_epochs=500)


/tmp/ipykernel_445631/3573039655.py:13: DeprecationWarning: /opt/conda/lib/python3.10/site-packages/ignite/contrib/handlers/param_scheduler.py has been moved to /ignite/handlers/param_scheduler.py and will be removed in version 0.6.0.
 Please refer to the documentation for more details.
  from ignite.contrib.handlers.param_scheduler import LRScheduler


Fold 1/10


/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Start training fold 1!
Fold 1 - Epoch 1 - Training Accuracy: 0.896, Loss: 0.415
Fold 1 - Epoch: 1 - Validation Accuracy: 0.928, Loss: 0.396
Learning rate atual: 0.001
Fold 1 - Epoch 2 - Training Accuracy: 0.912, Loss: 0.399
Fold 1 - Epoch: 2 - Validation Accuracy: 0.871, Loss: 0.428
Learning rate atual: 0.001
Fold 1 - Epoch 3 - Training Accuracy: 0.909, Loss: 0.404
Fold 1 - Epoch: 3 - Validation Accuracy: 0.849, Loss: 0.459
Learning rate atual: 0.001
Fold 1 - Epoch 4 - Training Accuracy: 0.918, Loss: 0.393
Fold 1 - Epoch: 4 - Validation Accuracy: 0.899, Loss: 0.402
Learning rate atual: 0.001
Fold 1 - Epoch 5 - Training Accuracy: 0.919, Loss: 0.394
Fold 1 - Epoch: 5 - Validation Accuracy: 0.928, Loss: 0.389
Learning rate atual: 0.001
Fold 1 - Epoch 6 - Training Accuracy: 0.933, Loss: 0.377
Fold 1 - Epoch: 6 - Validation Accuracy: 0.928, Loss: 0.391
Learning rate atual: 0.001
Fold 1 - Epoch 7 - Training Accuracy: 0.934, Loss: 0.376
Fold 1 - Epoch: 7 - Validation Accuracy: 0.863, Loss: 0.

2025-02-15 21:23:52,965 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


...

Fold 8 - Epoch: 106 - Validation Accuracy: 0.971, Loss: 0.339
Learning rate atual: 0.001
Fold 9/10
Start training fold 9!
Fold 9 - Epoch 1 - Training Accuracy: 0.876, Loss: 0.434
Fold 9 - Epoch: 1 - Validation Accuracy: 0.841, Loss: 0.462
Learning rate atual: 0.001
Fold 9 - Epoch 2 - Training Accuracy: 0.893, Loss: 0.418
Fold 9 - Epoch: 2 - Validation Accuracy: 0.833, Loss: 0.462
Learning rate atual: 0.001
Fold 9 - Epoch 3 - Training Accuracy: 0.923, Loss: 0.389
Fold 9 - Epoch: 3 - Validation Accuracy: 0.913, Loss: 0.395
Learning rate atual: 0.001
Fold 9 - Epoch 4 - Training Accuracy: 0.930, Loss: 0.382
Fold 9 - Epoch: 4 - Validation Accuracy: 0.913, Loss: 0.398
Learning rate atual: 0.001
Fold 9 - Epoch 5 - Training Accuracy: 0.925, Loss: 0.388
Fold 9 - Epoch: 5 - Validation Accuracy: 0.928, Loss: 0.386
Learning rate atual: 0.001
Fold 9 - Epoch 6 - Training Accuracy: 0.939, Loss: 0.373
Fold 9 - Epoch: 6 - Validation Accuracy: 0.870, Loss: 0.437
Learning rate atual: 0.001
Fold 9 - Epoch

2025-02-18 09:27:13,966 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Fold 9 - Epoch: 111 - Validation Accuracy: 0.935, Loss: 0.384
Learning rate atual: 0.001
Fold 10/10
Start training fold 10!
Fold 10 - Epoch 1 - Training Accuracy: 0.900, Loss: 0.408
Fold 10 - Epoch: 1 - Validation Accuracy: 0.899, Loss: 0.419
Learning rate atual: 0.001
Fold 10 - Epoch 2 - Training Accuracy: 0.917, Loss: 0.394
Fold 10 - Epoch: 2 - Validation Accuracy: 0.935, Loss: 0.377
Learning rate atual: 0.001
Fold 10 - Epoch 3 - Training Accuracy: 0.927, Loss: 0.383
Fold 10 - Epoch: 3 - Validation Accuracy: 0.891, Loss: 0.418
Learning rate atual: 0.001
Fold 10 - Epoch 4 - Training Accuracy: 0.919, Loss: 0.389
Fold 10 - Epoch: 4 - Validation Accuracy: 0.891, Loss: 0.404
Learning rate atual: 0.001
Fold 10 - Epoch 5 - Training Accuracy: 0.934, Loss: 0.377
Fold 10 - Epoch: 5 - Validation Accuracy: 0.935, Loss: 0.375
Learning rate atual: 0.001
Fold 10 - Epoch 6 - Training Accuracy: 0.915, Loss: 0.394
Fold 10 - Epoch: 6 - Validation Accuracy: 0.949, Loss: 0.367
Learning rate atual: 0.001


2025-02-18 15:40:23,729 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Fold 10 - Epoch: 92 - Validation Accuracy: 0.964, Loss: 0.349
Learning rate atual: 0.001
